[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/example/Grad_CAM/grad_cam.ipynb)

## Grad-CAM example

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
import os

repo_url = "https://github.com/geraldmc/irilab2026.git"
repo_name = "irilab2026" # The folder created when you clone

if not os.path.exists(repo_name):
    print(f"Cloning {repo_name}...")
    !git clone $repo_url
else:
    print(f"{repo_name} already exists. Skipping clone.")

In [ ]:
# 1. If necessary, remove the existing repository folder 
# (replace 'repo-name' with your actual folder name) and rerun above cell.
!rm -rf irilab2026

In [ ]:
# load some helper functions
%run /content/irilab2026/example/Grad_CAM/helper.py

In [ ]:
import torch
import torch.nn as nn
from torch.utils import data
from torchvision import models
from torchvision import transforms
from torchvision import datasets
import matplotlib.pyplot as plt
import numpy as np

# VGG19 is deep CNN architecture widely used for image recognition and classification. 
vgg19_model = models.vgg19(weights="DEFAULT")

# use the ImageNet transformation
transform = transforms.Compose([transforms.Resize((224, 224)), 
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
# check the image path 
def is_valid_image(path):
    # Only allow typical image extensions and skip hidden files
    valid_extensions = ('.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp')
    return path.lower().endswith(valid_extensions) and not os.path.basename(path).startswith('.')

# define a 5 image dataset
image_dir = "/content/irilab2026/example/Grad_CAM/my_dataset/"

dataset = datasets.ImageFolder(root=image_dir, is_valid_file=is_valid_image, transform=transform)
dataloader = data.DataLoader(dataset=dataset, shuffle=False, batch_size=5)


In [ ]:
# Run this to test your data loader
images, labels = next(iter(dataloader))
imshow(images[0], normalize=True)

In [ ]:
# We need to take a look at the VGG19 model's features, its architecture. 
# VGG19 is deep CNN architecture widely used for image recognition and classification. 
vgg19_model = models.vgg19(weights="DEFAULT")

vgg19_model.features

In [ ]:
class VGG(nn.Module):
    def __init__(self):
        super(VGG, self).__init__()
        
        # get the pretrained VGG19 network
        self.vgg = models.vgg19(weights="DEFAULT")
        
        # disect the network to access its last convolutional layer
        self.features_conv = self.vgg.features[:36]
        
        # get the max pool of the features stem
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        
        # get the classifier of the vgg19
        self.classifier = self.vgg.classifier
        
        # placeholder for the gradients
        self.gradients = None
    
    # hook for the gradients of the activations
    def activations_hook(self, grad):
        self.gradients = grad
        
    def forward(self, x):
        x = self.features_conv(x)
        
        # register the hook
        h = x.register_hook(self.activations_hook)
        
        # apply the remaining pooling
        x = self.max_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
    # method for the gradient extraction
    def get_activations_gradient(self):
        return self.gradients
    
    # method for the activation exctraction
    def get_activations(self, x):
        return self.features_conv(x)

In [ ]:
# initialize the VGG model
vgg = VGG()

# set the evaluation mode
vgg.eval()

# get the image from the dataloader
images, _ = next(iter(dataloader))

# keep the batch dimension: images[0:1] is (1, 3, 224, 224), not (3, 224, 224)
img = images[0:1]

# get the most likely prediction of the model
pred = vgg(img).argmax(dim=1)